# Embedder Comparison Notebook

**Goal**: compare the embedding model used in the paper
(`stsb-mpnet-base-v2`) with popular alternatives on the same
HotpotQA retrieval task used in `ChunkingTecniqueTesting.ipynb`.

## Models benchmarked

| Model | Dim | Notes |
|-------|-----|-------|
| `stsb-mpnet-base-v2` | 768 | **Paper's model** — STS fine-tuned MPNet |
| `all-mpnet-base-v2` | 768 | MPNet fine-tuned on 1B sentence pairs |
| `all-MiniLM-L6-v2` | 384 | Fast & lightweight (6-layer MiniLM) |
| `all-MiniLM-L12-v2` | 384 | More accurate MiniLM (12 layers) |
| `paraphrase-MiniLM-L6-v2` | 384 | Paraphrase-tuned MiniLM — different signal |
| `multi-qa-mpnet-base-dot-v1` | 768 | QA-optimised MPNet — closest to retrieval task |

## Evaluation
Same DSC chunking + FAISS top-5 retrieval + token-overlap relevance as in the
chunking notebook.  Metrics: **DCG@5, F1@5, P@5, R@5**.

Additional analysis: embedding dimension, L2 norm distribution, encode speed.


In [ ]:
# ── Cell 1: Install / imports ─────────────────────────────────────────────────
%pip install -q sentence-transformers faiss-cpu tiktoken datasets langchain-text-splitters tqdm ipywidgets seaborn

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from typing import List, Dict

import tiktoken
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

TOKENIZER_ENC = "cl100k_base"
TOP_K         = 5
MAX_TOKENS    = 100
MAX_EXAMPLES  = 200   # gold-standard queries to evaluate

_enc = tiktoken.get_encoding(TOKENIZER_ENC)
def count_tokens(text: str) -> int:
    return len(_enc.encode(text))

# ── Candidate embedders ────────────────────────────────────────────────────────
EMBEDDERS = {
    "stsb-mpnet-base-v2 (paper)":       "sentence-transformers/stsb-mpnet-base-v2",
    "all-mpnet-base-v2":                "sentence-transformers/all-mpnet-base-v2",
    "all-MiniLM-L6-v2":                 "sentence-transformers/all-MiniLM-L6-v2",
    "all-MiniLM-L12-v2":                "sentence-transformers/all-MiniLM-L12-v2",
    "paraphrase-MiniLM-L6-v2":          "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "multi-qa-mpnet-base-dot-v1":       "sentence-transformers/multi-qa-mpnet-base-dot-v1",
}

print("Embedders to compare:")
for k, v in EMBEDDERS.items():
    print(f"  {k:45s}  →  {v}")


In [ ]:
# ── Cell 2: Load HotpotQA & build corpus + gold-standard ─────────────────────
# Same logic as ChunkingTecniqueTesting.ipynb Cell 2.

from datasets import load_dataset
import difflib, re

print("Loading HotpotQA (distractor) ...")
dataset    = load_dataset("hotpot_qa", "distractor")
split_name = "validation" if "validation" in dataset else "test"
data_split = dataset[split_name]
print(f"Split: {split_name}  |  {len(data_split)} examples")

# Build per-example corpus docs
CORPUS_DOCS = []
for ex in data_split:
    ctx       = ex.get("context", {})
    ctx_sents = ctx.get("sentences", [])
    doc_text  = "\n\n".join(" ".join(sl) for sl in ctx_sents)
    CORPUS_DOCS.append({
        "id": ex.get("id", ""), "text": doc_text,
        "ctx_titles": ctx.get("title", []), "ctx_sents": ctx_sents,
    })

SEP = "\n\n===DOC===\n\n"
texts = [d["text"] for d in CORPUS_DOCS if d["text"].strip()]
doc_starts, offset = [], 0
for t in texts:
    doc_starts.append(offset)
    offset += len(t) + len(SEP)
corpus_text = SEP.join(texts)
print(f"Corpus: {len(texts)} docs  |  {len(corpus_text):,} chars")

# Build gold_standard
def _norm(s):
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

norm_corpus  = _norm(corpus_text)
gold_standard, skipped = [], 0

for ex in data_split:
    sup       = ex.get("supporting_facts", {})
    sf_titles = sup.get("title", [])   if isinstance(sup, dict) else []
    sf_ids    = sup.get("sent_id", []) if isinstance(sup, dict) else []
    if not sf_titles: skipped += 1; continue

    ctx        = ex.get("context", {})
    t2s        = dict(zip(ctx.get("title", []), ctx.get("sentences", [])))
    parts      = [t2s[t][i] for t, i in zip(sf_titles, sf_ids)
                  if t in t2s and 0 <= i < len(t2s[t])]
    if not parts: skipped += 1; continue

    gold_text  = " ".join(parts)
    norm_gold  = _norm(gold_text)
    prefix     = norm_gold[:120]
    found, char_start, char_end = False, 0, 0

    if prefix and prefix in norm_corpus:
        idx = corpus_text.lower().find(gold_text[:120].lower())
        if idx != -1:
            char_start, char_end, found = idx, min(idx + len(gold_text) + 200, len(corpus_text)), True

    if not found:
        for ds, d in zip(doc_starts, texts):
            if prefix and prefix in _norm(d):
                idx = d.lower().find(parts[0][:120].lower())
                if idx != -1:
                    char_start, char_end, found = ds + idx, min(ds + idx + len(gold_text) + 200, len(corpus_text)), True
                    break

    if not found: skipped += 1; continue

    gold_standard.append({"query": ex["question"],
                          "gold_char_start": char_start, "gold_char_end": char_end,
                          "gold_text": corpus_text[char_start:char_end]})
    if len(gold_standard) >= MAX_EXAMPLES:
        break

print(f"Gold-standard: {len(gold_standard)} queries  (skipped {skipped})")


In [ ]:
# ── Cell 3: Shared chunker (Fixed Token, same as paper baseline) ──────────────
# We use a single chunker so differences come purely from the embedder.

def fixed_token_chunker(text: str, max_tokens: int = MAX_TOKENS) -> List[str]:
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name=TOKENIZER_ENC,
        chunk_size=max_tokens,
        chunk_overlap=10,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)

chunks_shared = fixed_token_chunker(corpus_text)
print(f"Shared chunks: {len(chunks_shared)}  |  avg tokens: {np.mean([count_tokens(c) for c in chunks_shared]):.1f}")


In [ ]:
# ── Cell 4: Evaluation helpers ────────────────────────────────────────────────

def _token_set(text: str) -> set:
    return set(_enc.encode(text))

def dcg_at_k(relevances: List[int]) -> float:
    return sum(r / np.log2(rank + 2) for rank, r in enumerate(relevances))

def evaluate_embedder(model: SentenceTransformer,
                      chunks: List[str],
                      gold_data: List[Dict],
                      k: int = TOP_K) -> Dict:
    """
    Embed chunks with `model`, build FAISS index, retrieve top-k per query,
    compute DCG@k, F1@k, P@k, R@k.  Returns a metrics dict.
    """
    # Embed chunks
    t0 = time.time()
    vecs = model.encode(chunks, batch_size=64, show_progress_bar=False,
                        normalize_embeddings=True).astype(np.float32)
    embed_time = time.time() - t0

    dim   = vecs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vecs)

    dcg_s, f1_s, prec_s, rec_s = [], [], [], []
    q_vecs = model.encode([item["query"] for item in gold_data],
                           batch_size=64, show_progress_bar=False,
                           normalize_embeddings=True).astype(np.float32)

    for q_vec, item in zip(q_vecs, gold_data):
        gold_toks = _token_set(corpus_text[item["gold_char_start"]:item["gold_char_end"]])
        if not gold_toks:
            continue
        _, top_idx = index.search(q_vec.reshape(1, -1), k)
        top_idx    = top_idx[0]

        relevances = [1 if len(_token_set(chunks[ci]) & gold_toks) > 0 else 0
                      for ci in top_idx if 0 <= ci < len(chunks)]
        covered    = set().union(*[_token_set(chunks[ci]) & gold_toks
                                   for ci in top_idx if 0 <= ci < len(chunks)])

        prec = sum(relevances) / k
        rec  = len(covered) / len(gold_toks)
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

        dcg_s.append(dcg_at_k(relevances))
        prec_s.append(prec); rec_s.append(rec); f1_s.append(f1)

    l2_norms = np.linalg.norm(
        model.encode(chunks[:500], batch_size=64, show_progress_bar=False,
                     normalize_embeddings=False), axis=1)

    return dict(
        dcg       = float(np.mean(dcg_s))  if dcg_s  else 0,
        f1        = float(np.mean(f1_s))   if f1_s   else 0,
        precision = float(np.mean(prec_s)) if prec_s else 0,
        recall    = float(np.mean(rec_s))  if rec_s  else 0,
        dim       = dim,
        embed_time_s = embed_time,
        l2_mean   = float(np.mean(l2_norms)),
        l2_std    = float(np.std(l2_norms)),
    )

print("Evaluation helpers ready.")


In [ ]:
# ── Cell 5: Run comparison ────────────────────────────────────────────────────
# Loads each model, evaluates, collects metrics.

results = []
l2_plot_data = []

for name, model_id in tqdm(EMBEDDERS.items(), desc="Embedders"):
    print(f"\n  ► {name}  ({model_id})")
    model = SentenceTransformer(model_id)
    metrics = evaluate_embedder(model, chunks_shared, gold_standard, k=TOP_K)
    metrics["model"] = name
    results.append(metrics)

    # Collect L2 norms for distribution plot
    raw_vecs = model.encode(chunks_shared[:500], batch_size=64,
                             show_progress_bar=False, normalize_embeddings=False)
    for v in np.linalg.norm(raw_vecs, axis=1):
        l2_plot_data.append({"model": name, "l2_norm": float(v)})

    print(f"     dim={metrics['dim']}  DCG={metrics['dcg']:.3f}  "
          f"F1={metrics['f1']:.3f}  P@5={metrics['precision']:.3f}  "
          f"R@5={metrics['recall']:.3f}  t={metrics['embed_time_s']:.1f}s")
    del model   # free GPU/CPU memory between models

res_df = pd.DataFrame(results).set_index("model")
res_df = res_df[["dim", "dcg", "f1", "precision", "recall", "embed_time_s", "l2_mean", "l2_std"]]
res_df.columns = ["Dim", "DCG↑", "F1↑", "P@5↑", "R@5↑", "Enc time (s)", "L2 mean", "L2 std"]

print("\n── Results ──────────────────────────────────────────────────────────────")
print(res_df.round(4).to_string())


In [ ]:
# ── Cell 6: Retrieval-metric bar charts ───────────────────────────────────────

metrics_to_plot = [("DCG↑", "DCG@5"), ("F1↑", "F1@5"),
                   ("P@5↑", "Precision@5"), ("R@5↑", "Recall@5")]
colors = sns.color_palette("tab10", n_colors=len(res_df))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Embedder Comparison — HotpotQA retrieval (Fixed Token chunks, K=5)",
             fontsize=13, y=1.02)

for ax, (col, label) in zip(axes, metrics_to_plot):
    vals    = res_df[col].values
    models  = res_df.index.tolist()
    bars    = ax.bar(models, vals, color=colors, edgecolor="white")
    ax.set_title(label, fontsize=11)
    ax.set_ylabel(col)
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    ax.tick_params(axis="x", rotation=35, labelsize=7)
    # Highlight paper model
    for bar, m, v in zip(bars, models, vals):
        lw = 2.5 if "paper" in m else 0
        bar.set_edgecolor("black" if "paper" in m else "white")
        bar.set_linewidth(lw)
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig("embedder_comparison_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved embedder_comparison_metrics.png")


In [ ]:
# ── Cell 7: Speed vs. quality scatter ─────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
for i, (name, row) in enumerate(res_df.iterrows()):
    is_paper = "paper" in name
    ax.scatter(row["Enc time (s)"], row["F1↑"],
               color=colors[i], s=120,
               marker="*" if is_paper else "o",
               zorder=3, label=name)
    ax.annotate(name, (row["Enc time (s)"], row["F1↑"]),
                textcoords="offset points", xytext=(6, 2), fontsize=8)

ax.set_xlabel("Encoding time (s) for all chunks")
ax.set_ylabel("F1@5")
ax.set_title("Speed vs. retrieval quality — embedder comparison\n(★ = paper model)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("embedder_speed_vs_quality.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved embedder_speed_vs_quality.png")


In [ ]:
# ── Cell 8: Embedding L2-norm distribution (raw, unnormalised) ────────────────

plt.figure(figsize=(13, 5))
sns.boxplot(x="model", y="l2_norm", data=pd.DataFrame(l2_plot_data),
            palette="tab10", order=list(EMBEDDERS.keys()))
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.title("Per-chunk L2-norm distribution (raw embeddings, first 500 chunks)")
plt.ylabel("L2 norm")
plt.tight_layout()
plt.savefig("embedder_l2_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved embedder_l2_distribution.png")


In [ ]:
# ── Cell 9: Summary table with rank ───────────────────────────────────────────

summary = res_df[["Dim", "DCG↑", "F1↑", "P@5↑", "R@5↑", "Enc time (s)"]].copy()
summary["F1 rank"] = summary["F1↑"].rank(ascending=False).astype(int)
summary["DCG rank"] = summary["DCG↑"].rank(ascending=False).astype(int)
summary = summary.sort_values("F1 rank")

# Highlight the paper's model row
def highlight_paper(row):
    return ["font-weight: bold; background-color: #ffffcc"
            if "paper" in row.name else "" for _ in row]

display(summary.round(4).style.apply(highlight_paper, axis=1).set_caption(
    "Embedder benchmark — bold/yellow = paper model (stsb-mpnet-base-v2)"))
